1) Initially study again Lesk implementation in NLTK. Consider the sentence “I have been prescribed two
important drugs today during my visit to clinic” and assume that the target word is “drug”. Construct a
simple script that outputs the correct sense of “drug” in the above sentence using standard Lesk
algorithm. And, comment on the outcome the efficiency of the result.

These are all the needed packages to implement the Lesk algorithm

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.wsd import lesk
from nltk.corpus import wordnet as wn

# Downloading all needed packages
for pkg in packages:
    nltk.download(pkg, quiet=True)

Implementing now the Lesk algorithm with the specified sentence and the given target word

In [ ]:

packages = ("wordnet", "omw-1.4", "punkt", "punkt_tab")
sentence = "I have been prescribed two important drugs today during my visit to clinic"
target_word = "drug"

tokens = word_tokenize(sentence)

for s in wn.synsets(target_word):
    print(f"  {s.name():<12} [{s.pos()}] {s.definition()}")

# Standard Lesk, no POS restriction
s1 = lesk(tokens, target_word)
print("\nLesk (no POS):", s1, "->", s1.definition())

# Standard Lesk, restricted to nouns
s2 = lesk(tokens, target_word, pos="n")
print("Lesk (pos='n'):", s2, "->", s2.definition())

  drug.n.01    [n] a substance that is used as a medicine or narcotic
  drug.v.01    [v] administer a drug to
  drug.v.02    [v] use recreational drugs

Lesk (no POS): Synset('drug.v.01') -> administer a drug to
Lesk (pos='n'): Synset('drug.n.01') -> a substance that is used as a medicine or narcotic


Standard Lesk algorithm finds three meanings of the word "drug". Two are verbs and one is a noun. While we perfectly understand that in that specific case, the right meaning is the one of the noun, lesk algorithm thinks that the more accurate one is the verb. This happens because of a well known weakness of Lesk algorithm: it doesn't perform well when the terms doesn't appear in both the context and the gloss. Indeed, when we limit the search only to the nouns, the algorithm finds out the real meaning of the word "drug" (even though we had to 'force' the real result). To improve the results, we should try to use POS tagging or/and extended/adapted Lesk.

Let's try the new effective version of the task

In [ ]:
import nltk
from nltk import pos_tag
from nltk.corpus import stopwords, wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.wsd import lesk

for pkg in ("wordnet", "omw-1.4", "punkt", "punkt_tab",
            "averaged_perceptron_tagger_eng", "stopwords"):
    nltk.download(pkg, quiet=True)

SENTENCE = "I have been prescribed two important drugs today during my visit to clinic"
TARGET = "drug"

lemmatizer = WordNetLemmatizer()
STOP = set(stopwords.words("english"))


# ---------------------------------------------------------------------------
# 1) POS tagging: Penn Treebank tag -> WordNet POS
# ---------------------------------------------------------------------------
def penn_to_wn(tag):
    if tag.startswith("J"):
        return wn.ADJ
    if tag.startswith("V"):
        return wn.VERB
    if tag.startswith("N"):
        return wn.NOUN
    if tag.startswith("R"):
        return wn.ADV
    return None


def find_target(tagged, target):
    """Return (index, token, wordnet_pos) of the token whose lemma is `target`."""
    for i, (tok, tag) in enumerate(tagged):
        pos = penn_to_wn(tag)
        if pos and lemmatizer.lemmatize(tok.lower(), pos) == target:
            return i, tok, pos
    raise ValueError(f"'{target}' not found")


# 2) Extended / Adapted Lesk  
def related_synsets(s, depth=2):
    """Related synsets; hyponyms are followed `depth` levels down."""
    rel = []
    rel += s.hypernyms()
    level = s.hyponyms()
    for _ in range(depth):
        rel += level
        level = [h for x in level for h in x.hyponyms()]
    rel += s.part_meronyms() + s.member_meronyms() + s.substance_meronyms()
    rel += s.part_holonyms() + s.member_holonyms() + s.substance_holonyms()
    rel += s.similar_tos() + s.also_sees() + s.attributes()
    rel += s.entailments() + s.verb_groups()
    return rel


def normalize(text):
    """Lowercase, tokenize, drop stopwords/punctuation, lemmatize."""
    out = []
    for w in word_tokenize(text.lower()):
        if w.isalpha() and w not in STOP:
            out.append(lemmatizer.lemmatize(w))
    return out


def extended_gloss(s):
    words = normalize(s.definition())
    for ex in s.examples():
        words += normalize(ex)
    for r in related_synsets(s):
        words += [l.lower() for name in r.lemma_names() for l in name.split("_")]
        words += normalize(r.definition())
        for ex in r.examples():
            words += normalize(ex)
    return words


def context_lemmas(tagged, idx):
    """Lemmas of the context words, using their own POS tag (prescribed -> prescribe)."""
    out = set()
    for i, (tok, tag) in enumerate(tagged):
        if i == idx or tok.lower() in STOP or not tok.isalpha():
            continue
        out.add(lemmatizer.lemmatize(tok.lower(), penn_to_wn(tag) or wn.NOUN))
    return out


def extended_lesk(tagged, idx, pos=None):
    """Return (best_synset, scores, tie). Falls back to the first (most frequent)
    WordNet sense when no sense has any overlap, and says so."""
    tok = tagged[idx][0]
    lemma = lemmatizer.lemmatize(tok.lower(), pos or wn.NOUN)
    context = context_lemmas(tagged, idx)
    scores = {}
    for s in wn.synsets(lemma, pos=pos):
        overlap = context & set(extended_gloss(s))
        scores[s] = (len(overlap), sorted(overlap))
    top = max(n for n, _ in scores.values())
    winners = [s for s, (n, _) in scores.items() if n == top]
    return winners[0], scores, (top == 0 or len(winners) > 1)


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
tokens = word_tokenize(SENTENCE)
tagged = pos_tag(tokens)
idx, tok, wnpos = find_target(tagged, TARGET)

print("POS tags :", tagged)
print(f"Target   : '{tok}' (index {idx}) -> WordNet POS = '{wnpos}'\n")

# (a) Standard NLTK Lesk, no POS
s = lesk(tokens, tok)
print("[a] Standard Lesk, no POS     :", s, "-", s.definition())

# (b) Standard NLTK Lesk + POS tagging
s = lesk(tokens, tok, pos=wnpos)
print("[b] Standard Lesk + POS       :", s, "-", s.definition())

# (c) Extended Lesk, no POS filter
s, scores_all, tie = extended_lesk(tagged, idx)
print("[c] Extended Lesk, no POS     :", s, "-", s.definition(), "(TIE / no evidence)" if tie else "")

# (d) Extended Lesk + POS tagging
s, scores, tie = extended_lesk(tagged, idx, pos=wnpos)
print("[d] Extended Lesk + POS       :", s, "-", s.definition(), "(TIE / no evidence)" if tie else "")

print("\nExtended Lesk overlap scores (no POS filter):")
for syn, (n, ov) in scores_all.items():
    print(f"  {syn.name():<12} score={n:<3} overlap={ov}")

POS tags : [('I', 'PRP'), ('have', 'VBP'), ('been', 'VBN'), ('prescribed', 'VBN'), ('two', 'CD'), ('important', 'JJ'), ('drugs', 'NNS'), ('today', 'NN'), ('during', 'IN'), ('my', 'PRP$'), ('visit', 'NN'), ('to', 'TO'), ('clinic', 'VB')]
Target   : 'drugs' (index 6) -> WordNet POS = 'n'

[a] Standard Lesk, no POS     : Synset('drug.v.01') - administer a drug to
[b] Standard Lesk + POS       : Synset('drug.n.01') - a substance that is used as a medicine or narcotic
[c] Extended Lesk, no POS     : Synset('drug.n.01') - a substance that is used as a medicine or narcotic 
[d] Extended Lesk + POS       : Synset('drug.n.01') - a substance that is used as a medicine or narcotic 

Extended Lesk overlap scores (no POS filter):
  drug.n.01    score=1   overlap=['two']
  drug.v.01    score=0   overlap=[]
  drug.v.02    score=0   overlap=[]
